# Lesson 7 · Wave–particle duality and the double slit

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/StativeSh/quantum-playground/blob/main/lessons/07_wave_particle_duality.ipynb)

**For:** A-level Physics (for example AQA 3.2.2 and 3.3.2.1, OCR A 4.4.3 and 4.5, Edexcel Topic 5) and IB Physics (C.3 wave phenomena and E.2 quantum physics). Check your own specification for the exact wording.

**Before this lesson:** Lesson 4, *Stationary waves and harmonics* (superposition and path difference); Lesson 5, *Electric fields and potential* (energy gained moving through a potential difference, W = qV). Also assumed: the electromagnetic spectrum and the electronvolt.

**Leads to:** Lesson 15, *Diffraction gratings and optics*.

**You will be able to:**

1. explain the photoelectric effect with photons, and use $hf = \phi + E_{k,max}$
2. use $s = \lambda D/d$ for Young's double slit, and describe the effect of slit width
3. use the de Broglie relation $\lambda = h/p$, including for electrons accelerated through a p.d.
4. explain electron diffraction as evidence that particles behave as waves
5. interpret $|\psi|^2$ as probability, and explain how an interference pattern builds up one particle at a time

**How to use this lesson:** run each cell with Shift+Enter. Where you see *Try it*, change a number and run the cell again. Before some experiments a **Predict** box asks you to commit to an answer; write it down, run the cell, then answer the **Explain** box below it. Answers to both are at the very end, after the worked solutions. Exercises are at the end, with worked solutions below them.

In [ ]:
# Install the course API in a fresh Colab runtime or upgrade an older installation.
import importlib.util
import subprocess
import sys
from importlib.metadata import version

if importlib.util.find_spec("qplay") is None or int(version("quantum-playground").split(".")[0]) < 1:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "quantum-playground>=1,<2"], check=True)

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from qplay import quantum

h = 6.626e-34         # J s
c = 3.00e8            # m/s
e = 1.602e-19         # C
m_e = 9.109e-31       # kg
m_n = 1.675e-27       # kg

## 1. Light as particles: the photoelectric effect

Shine light on a clean metal and electrons can be knocked out, but only if the light's **frequency** is above a threshold. Below it, no electrons come out however bright the light. A wave picture cannot explain this; Einstein's photon picture can. Light arrives in packets of energy

$$E = hf,$$

and one photon gives all its energy to one electron. The electron needs at least the **work function** $\phi$ to escape, so

$$hf = \phi + E_{k,max}\qquad\Rightarrow\qquad E_{k,max} = hf - \phi,\qquad f_0 = \frac{\phi}{h}.$$

In [ ]:
metals = {"caesium": 2.1, "sodium": 2.28, "zinc": 4.3, "platinum": 5.6}    # work functions in eV
f = np.linspace(0, 2.0e15, 400)
plt.figure(figsize=(7, 4))
for i, (name, phi) in enumerate(metals.items()):
    Ek = h * f / e - phi
    plt.plot(f / 1e15, np.where(Ek > 0, Ek, np.nan), color=f"C{i}", label=f"{name}: f0 = {phi * e / h / 1e15:.2f} x 10^15 Hz")
    plt.plot(f / 1e15, Ek, color=f"C{i}", ls=":", lw=0.8)
plt.axhline(0, color="k", lw=0.8)
for lam, colour in [(700e-9, "red"), (400e-9, "violet")]:
    plt.axvline(c / lam / 1e15, color=colour, lw=6, alpha=0.25)
plt.ylim(-6, 6); plt.xlabel("frequency / 10$^{15}$ Hz"); plt.ylabel("maximum kinetic energy / eV"); plt.legend(fontsize=8)
plt.title(f"Every line has gradient h = {h / e:.3e} eV s");

<a id="young-fringe-spacing"></a>

All the lines are parallel, with gradient $h$ (Planck's constant), and each meets the frequency axis at its own threshold $f_0$. Extended backwards (dotted), each line meets the energy axis at $-\phi$. The shaded bands show visible light: it ejects electrons from caesium and sodium but not from zinc or platinum.

**Exam tip:** brighter light means *more photons per second*, so more electrons per second, but **not** faster electrons. Only the frequency sets $E_{k,max}$.

## 2. Light as waves: Young's double slit

Light through two narrow slits a distance $d$ apart makes bright and dark fringes on a screen a distance $D$ away. Bright fringes are where the **path difference** is a whole number of wavelengths. For small angles the spacing is

$$s = \frac{\lambda D}{d}.$$

Each slit also has a width $w$, which spreads its light out (single-slit diffraction) and sets an overall envelope. The intensity across the screen is

$$I(y) = I_0\cos^2\!\left(\frac{\pi d y}{\lambda D}\right)\,\mathrm{sinc}^2\!\left(\frac{\pi w y}{\lambda D}\right).$$

In [ ]:
d, w, D = 0.25e-3, 0.04e-3, 1.0                 # slit separation, slit width, screen distance (m)   <- Try it
y = np.linspace(-0.03, 0.03, 3000)
plt.figure(figsize=(9, 3.6))
for lam, colour in [(650e-9, "red"), (450e-9, "blue")]:
    I = np.cos(np.pi * d * y / (lam * D)) ** 2 * np.sinc(w * y / (lam * D)) ** 2   # np.sinc(x) = sin(pi x)/(pi x)
    plt.plot(y * 1000, I, color=colour, lw=1, label=f"{lam * 1e9:.0f} nm: s = {lam * D / d * 1000:.2f} mm")
plt.xlabel("position on screen / mm"); plt.ylabel("intensity"); plt.legend(); plt.title("Longer wavelength, wider fringes");

In [ ]:
# Repeat the explorer measurement from the peaks of the intensity curve.
from scipy.signal import find_peaks

lam = 532e-9
screen = np.linspace(0, 0.006, 6001)  # metres from the central bright fringe
spacings = []
for separation in (0.25e-3, 0.50e-3):
    sin_theta = screen / np.hypot(screen, D)
    intensity = np.cos(np.pi * separation * sin_theta / lam)**2 * np.sinc(w * sin_theta / lam)**2
    peaks, _ = find_peaks(intensity)
    spacing = screen[peaks[0]]
    spacings.append(spacing)
    print(f"d = {separation*1000:.2f} mm: measured {spacing*1000:.3f} mm, predicted {lam*D/separation*1000:.3f} mm")
print(f"Measured spacing ratio: {spacings[1]/spacings[0]:.3f}; predicted: 0.500")

So light behaves as a wave when it travels and interferes, and as particles when it is absorbed. This is **wave–particle duality**.

## 3. Matter as waves: the de Broglie wavelength

In 1924 de Broglie suggested that the duality works the other way too: any particle with momentum $p$ has a wavelength

$$\lambda = \frac{h}{p} = \frac{h}{mv}.$$

An electron accelerated from rest through a potential difference $V$ gains kinetic energy $eV = p^2/2m$, so $\lambda = \dfrac{h}{\sqrt{2meV}}$. For a few hundred volts this is about the spacing between atoms in a crystal, so electrons fired at a crystal diffract, exactly as X-rays do. That was seen in 1927 and confirmed the idea.

**Predict (P1).** Which has the longer de Broglie wavelength: an electron moving at 1% of the speed of light, or a slow neutron at 2200 m s$^{-1}$? (The neutron is about 1800 times heavier.) Write your prediction down before you run the next cell.

In [ ]:
V = np.logspace(1, 5, 200)
lam = h / np.sqrt(2 * m_e * e * V)
plt.figure(figsize=(7, 3.8))
plt.loglog(V, lam * 1e9)
plt.axhspan(0.1, 0.4, color="C2", alpha=0.2, label="spacing of atoms in crystals")
plt.xlabel("accelerating voltage / V"); plt.ylabel("de Broglie wavelength / nm"); plt.legend()
plt.title("Electrons: wavelength against accelerating voltage")
for thing, mass, speed in [("electron at 1% of c", m_e, 3e6), ("thermal neutron", m_n, 2200), ("cricket ball", 0.16, 40)]:
    print(f"{thing:22s} lambda = {h / (mass * speed):.2e} m")

**Explain (P1).** Why are the two wavelengths so close, despite the huge difference in mass?

A cricket ball's wavelength is around $10^{-34}$ m, absurdly smaller than anything it could pass through, which is why everyday objects never show wave behaviour.

## 4. One electron through two slits

The package can solve the Schrödinger equation for a single particle (a wavepacket) passing through a wall with two slits (in units where $\hbar = m = 1$). The quantity $|\psi|^2$ is the **probability density**: where the particle is likely to be found. We add up $|\psi|^2$ on a screen over the whole run and compare the dark fringes with the path-difference rule: dark wherever the distances from the two slits differ by $(n + \tfrac12)\lambda$.

In [ ]:
k0, separation = 3.0, 5.0                         # wavenumber and slit separation (natural units)
run = quantum.double_slit_experiment(k0=k0, separation=separation)
lam_q = 2 * np.pi / k0                            # de Broglie wavelength, lambda = h/p = 2*pi/k
D_q = run["screen_x"]                             # wall at x = 0, screen at x = D

def path_difference(y):
    return np.hypot(D_q, y + separation / 2) - np.hypot(D_q, y - separation / 2)

# Dark where the path difference is (n + 1/2) wavelengths; it can never exceed the slit separation.
orders = [n for n in range(10) if (n + 0.5) * lam_q < path_difference(40)]
dark = [brentq(lambda y, n=n: path_difference(y) - (n + 0.5) * lam_q, 0, 40) for n in orders]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1.3, 1]})
frame = len(run["times"]) * 3 // 4
ax1.imshow(np.abs(run["psi"][frame]) ** 0.5, origin="lower", cmap="magma",
           extent=[run["x"][0], run["x"][-1], run["y"][0], run["y"][-1]])
ax1.contour(run["x"], run["y"], run["V"], [1], colors="c", linewidths=0.8)
ax1.axvline(D_q, color="w", ls="--", lw=0.8)
ax1.set(xlim=(-15, 45), ylim=(-30, 30), title=f"|psi| at t = {run['times'][frame]:.1f} (screen dashed)")
ax1.grid(False)
ax2.plot(run["y"], run["screen"], label="simulated, summed over time")
for n, yd in enumerate(dark):
    ax2.axvline(yd, color="C3", ls=":", lw=1, label="path difference = (n + 1/2) wavelengths" if n == 0 else None)
    ax2.axvline(-yd, color="C3", ls=":", lw=1)
ax2.set(xlim=(-25, 25), xlabel="position on screen y", title="Pattern on the screen"); ax2.legend(fontsize=8)
print(f"wavelength = {lam_q:.3f};  predicted dark fringes at y = " + ", ".join(f"±{yd:.2f}" for yd in dark))

The simulated dark fringes sit where the path-difference rule puts them. (There are only two on each side here: the path difference can never be larger than the slit separation, 5, and $2.5\lambda = 5.2$.) A single particle, described by a wave that goes through **both** slits, interferes with itself.

## 5. Building up the pattern one particle at a time

So does each electron arrive as a spread-out wave? No: each one is detected at a single point, like a particle. $|\psi|^2$ only says where it is *likely* to land. Below we let the computer pick random landing points with probabilities given by the screen pattern above.

**Predict (P2).** Electrons arrive one at a time. Roughly how many must arrive before you can clearly see fringes: 20, 200, 2000 or 20 000? Write your prediction down before you run the next cell.

In [ ]:
rng = np.random.default_rng(1)
p = run["screen"] / run["screen"].sum()
fig, axes = plt.subplots(1, 4, figsize=(13, 3.6), sharey=True)
for ax, n in zip(axes, [20, 200, 2000, 20000]):              # <- Try it: other numbers
    hits = rng.choice(run["y"], size=n, p=p) + rng.uniform(-0.15, 0.15, n)
    ax.plot(rng.uniform(0, 1, n), hits, ".", ms=max(0.6, 5 / np.sqrt(n / 20)), color="k", alpha=0.6)
    ax.set(title=f"{n} electrons", xticks=[], ylim=(-25, 25))
axes[0].set_ylabel("position on screen y");

**Explain (P2).** What does a single electron's landing point tell you, and what does the pattern of many tell you?

With 20 electrons the hits look random. By 20 000 the fringes are unmistakable. This was done for real with electrons sent one at a time (Tonomura and colleagues, 1989): each arrives as a particle, and the pattern of many is a wave's.

---

## Exercises

Take $h = 6.63\times10^{-34}$ J s, $c = 3.00\times10^{8}\ \mathrm{m\,s^{-1}}$, $e = 1.60\times10^{-19}$ C, $m_e = 9.11\times10^{-31}$ kg, $m_n = 1.68\times10^{-27}$ kg. Give answers to 3 significant figures.

**Q1.** Light of wavelength 633 nm passes through two slits 0.25 mm apart onto a screen 2.0 m away. Find the fringe spacing.

**Q2.** Fringes 3.2 mm apart are seen on a screen 1.8 m from slits 0.40 mm apart. Find the wavelength.

**Q3.** Find the de Broglie wavelength of an electron accelerated from rest through 150 V.

**Q4.** At what speed does a neutron have a de Broglie wavelength of 0.18 nm?

**Q5.** Zinc has a work function of 4.3 eV. Ultraviolet light of wavelength 200 nm falls on it. Find (a) the maximum kinetic energy of the emitted electrons, in eV, (b) the threshold frequency for zinc.

**Q6 (simulation).** For the simulation in section 4 ($\lambda = 2\pi/3$, slits 5.0 apart, screen 30 away), the small-angle formula gives the first dark fringe at $y = \lambda D/2d$. Find where it really is, using the exact path difference, and compare.

In [ ]:
def check(name, yours, correct, tol=0.01):
    if yours is None:
        print(f"{name}: not answered yet")
    elif abs(yours - correct) <= tol * abs(correct):
        print(f"{name}: correct")
    else:
        print(f"{name}: not yet, check your working (you gave {yours})")

# The correct values are computed here from the physics, not typed in.
_lam6 = 2 * np.pi / 3
_dark6 = brentq(lambda y: np.hypot(30, y + 2.5) - np.hypot(30, y - 2.5) - _lam6 / 2, 0, 20)
_answers = {
    "Q1": 633e-9 * 2.0 / 0.25e-3,
    "Q2": 3.2e-3 * 0.40e-3 / 1.8,
    "Q3": h / np.sqrt(2 * m_e * e * 150),
    "Q4": h / (m_n * 0.18e-9),
    "Q5a": h * c / 200e-9 / e - 4.3, "Q5b": 4.3 * e / h,
    "Q6": _dark6,
}

# Replace None with your answers, in SI units (Q5a in eV, Q6 in simulation units).
mine = {"Q1": None, "Q2": None, "Q3": None, "Q4": None, "Q5a": None, "Q5b": None, "Q6": None}
for q in mine:
    check(q, mine[q], _answers[q])

---

## Worked solutions

Try the questions before reading on.

**Q1.** $s = \dfrac{\lambda D}{d} = \dfrac{633\times10^{-9} \times 2.0}{0.25\times10^{-3}} = 5.06\times10^{-3}\ \mathrm{m}$, about 5.1 mm.

**Q2.** $\lambda = \dfrac{sd}{D} = \dfrac{3.2\times10^{-3} \times 0.40\times10^{-3}}{1.8} = 7.11\times10^{-7}\ \mathrm{m} = 711$ nm (red light).

**Q3.** $\lambda = \dfrac{h}{\sqrt{2meV}} = \dfrac{6.63\times10^{-34}}{\sqrt{2 \times 9.11\times10^{-31} \times 1.60\times10^{-19} \times 150}} = 1.00\times10^{-10}\ \mathrm{m}$, about the size of an atom.

**Q4.** $v = \dfrac{h}{m\lambda} = \dfrac{6.63\times10^{-34}}{1.68\times10^{-27} \times 0.18\times10^{-9}} = 2.20\times10^{3}\ \mathrm{m\,s^{-1}}$. Neutrons this slow are called thermal neutrons, and are used to study crystal structures.

**Q5.** (a) Photon energy $hc/\lambda = 9.94\times10^{-19}\ \mathrm{J} = 6.20$ eV, so $E_{k,max} = 6.20 - 4.3 = 1.90$ eV &nbsp; (b) $f_0 = \phi/h = 4.3 \times 1.60\times10^{-19}/6.63\times10^{-34} = 1.04\times10^{15}\ \mathrm{Hz}$ (ultraviolet).

**Q6.** The small-angle formula gives $y = \lambda D/2d = 2.094 \times 30/10 = 6.28$. Solving $\sqrt{D^2 + (y + d/2)^2} - \sqrt{D^2 + (y - d/2)^2} = \lambda/2$ exactly gives $y = 6.45$: the formula is about 3% low, because the angle (about 12°) is not that small. That is why the section 4 code uses the exact path difference.

In [ ]:
for q, value in _answers.items():
    print(f"{q}: {value:#.3g}")
print(f"small-angle formula: {_lam6 * 30 / (2 * 5):.3g}")

**Next:** in section 4, change `separation` to 8. Predict whether the fringes get closer together or further apart before you run it.

### Predict and explain: answers

**P1.** The electron, but only just: $2.4\times10^{-10}$ m against $1.8\times10^{-10}$ m. Wavelength depends on momentum $mv$, not on mass alone. The neutron is about 1840 times heavier but about 1360 times slower, so the two momenta are similar.

**P2.** Somewhere between a few hundred and two thousand: at 20 the hits look random, at 200 bands begin to show, and by 2000 they are clear. Each electron lands at one point, like a particle, at a random place; $|\psi|^2$ only sets the probabilities. The fringes are a statistical pattern, so they need many particles to appear.